In [1]:
# ============================================================
# Step 1: Build a DORA ligand library from PubChem
# ============================================================
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pubchempy as pcp
import pandas as pd
import requests
import time

In [2]:

# ============================================================
# 1a. Define seed compounds
# ============================================================

dora_names    = ["Daridorexant", "Lemborexant", "Suvorexant"]
agonist_names = ["Oveporexton"]


In [3]:


# ============================================================
# 1b. Fetch a single compound by name
# ============================================================

def fetch_compound_properties(name, drug_class):
    """
    Look up a compound by name on PubChem.
    Returns a property dictionary, or None if not found.
    """
    results = pcp.get_compounds(name, 'name')

    if not results:
        print(f"  WARNING: '{name}' not found on PubChem.")
        return None

    compound = results[0]

    return {
        "name":              name,
        "drug_class":        drug_class,
        "cid":               compound.cid,
        "molecular_formula": compound.molecular_formula,
        "molecular_weight":  compound.molecular_weight,
        "iupac_name":        compound.iupac_name,
        "canonical_smiles":  compound.connectivity_smiles,  # updated name
        "isomeric_smiles":   compound.smiles,               # updated name
        "xlogp":             compound.xlogp,
        "h_bond_donors":     compound.h_bond_donor_count,
        "h_bond_acceptors":  compound.h_bond_acceptor_count,
        "rotatable_bonds":   compound.rotatable_bond_count,
        "tpsa":              compound.tpsa,
        "exact_mass":        compound.exact_mass,
        "charge":            compound.charge,
    }


In [4]:
def fetch_compound_properties(name, drug_class):
    """
    Look up a compound by name on PubChem.
    Explicitly requests all properties to avoid None returns.
    """
    results = pcp.get_compounds(
        name, 
        'name',
        properties=[                      # tell PubChem exactly what to return
            'MolecularFormula',
            'MolecularWeight',
            'IsomericSMILES',
            'IUPACName',
            'XLogP',
            'HBondDonorCount',
            'HBondAcceptorCount',
            'RotatableBondCount',
            'TPSA',
            'ExactMass',
            'Charge',
        ]
    )

    if not results:
        print(f"  WARNING: '{name}' not found on PubChem.")
        return None

    compound = results[0]

    # Use .get() style access via the compound's to_dict() method
    # which is more robust than attribute access when fields are missing
    d = compound.to_dict()

    return {
        "name":              name,
        "drug_class":        drug_class,
        "cid":               compound.cid,
        "molecular_formula": d.get("molecular_formula"),
        "molecular_weight":  d.get("molecular_weight"),
        "iupac_name":        d.get("iupac_name"),
        "canonical_smiles":  d.get("isomeric_smiles"),   # use isomeric as canonical
        "isomeric_smiles":   d.get("isomeric_smiles"),
        "xlogp":             d.get("xlogp"),
        "h_bond_donors":     d.get("h_bond_donor_count"),
        "h_bond_acceptors":  d.get("h_bond_acceptor_count"),
        "rotatable_bonds":   d.get("rotatable_bond_count"),
        "tpsa":              d.get("tpsa"),
        "exact_mass":        d.get("exact_mass"),
        "charge":            d.get("charge"),
    }

In [5]:
# 1c. Fetch all seed compounds
records = []

print("Fetching seed compounds from PubChem...")

for name in dora_names:
    print(f"  Searching: {name}")
    data = fetch_compound_properties(name, drug_class="DORA")
    if data:
        records.append(data)

for name in agonist_names:
    print(f"  Searching: {name}")
    data = fetch_compound_properties(name, drug_class="agonist")
    if data:
        records.append(data)

print(f"Seed compounds found: {len(records)}")

Fetching seed compounds from PubChem...
  Searching: Daridorexant
  Searching: Lemborexant
  Searching: Suvorexant
  Searching: Oveporexton
Seed compounds found: 4


In [6]:


# ============================================================
# 1d. Similarity search — with retry logic for network hiccups
# ============================================================

def find_similar_compounds(cid, drug_class, threshold=85,
                            max_results=20, max_retries=3):
    """
    Uses PubChem REST API directly to find structurally similar compounds.
    Retries automatically if the network drops temporarily.
    """
    print(f"  Similarity search for CID {cid}...")

    search_url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/"
        f"fastsimilarity_2d/cid/{cid}/cids/JSON"
        f"?Threshold={threshold}&MaxRecords={max_results}"
    )

    # --- Step 1: get the list of similar CIDs, with retries ---
    similar_cids = []
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(search_url, timeout=30)
            response.raise_for_status()
            similar_cids = (response.json()
                            .get("IdentifierList", {})
                            .get("CID", []))
            break  # success — exit the retry loop
        except Exception as e:
            print(f"    Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                wait = attempt * 3   # wait 3s, then 6s, then 9s
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"    All {max_retries} attempts failed. Skipping CID {cid}.")
                return []

    if not similar_cids:
        print(f"    No similar compounds found.")
        return []

    print(f"    Found {len(similar_cids)} similar CIDs. Fetching properties...")

    # --- Step 2: fetch properties for all similar CIDs in one batch ---
    cid_str  = ",".join(str(c) for c in similar_cids)
    props_url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}"
        f"/property/MolecularFormula,MolecularWeight,IsomericSMILES,"
        f"IUPACName,XLogP,HBondDonorCount,HBondAcceptorCount,"
        f"RotatableBondCount,TPSA,ExactMass,Charge/JSON"
    )

    for attempt in range(1, max_retries + 1):
        try:
            time.sleep(0.5)   # polite pause between requests
            response = requests.get(props_url, timeout=30)
            response.raise_for_status()
            props_list = (response.json()
                          .get("PropertyTable", {})
                          .get("Properties", []))
            break
        except Exception as e:
            print(f"    Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                wait = attempt * 3
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"    Could not fetch properties. Skipping CID {cid}.")
                return []

    hits = []
    for p in props_list:
        hits.append({
            "name":              f"similar_to_{cid}_{p['CID']}",
            "drug_class":        drug_class + "_analog",
            "cid":               p["CID"],
            "molecular_formula": p.get("MolecularFormula"),
            "molecular_weight":  p.get("MolecularWeight"),
            "iupac_name":        p.get("IUPACName"),
            "canonical_smiles":  p.get("IsomericSMILES"),
            "isomeric_smiles":   p.get("IsomericSMILES"),
            "xlogp":             p.get("XLogP"),
            "h_bond_donors":     p.get("HBondDonorCount"),
            "h_bond_acceptors":  p.get("HBondAcceptorCount"),
            "rotatable_bonds":   p.get("RotatableBondCount"),
            "tpsa":              p.get("TPSA"),
            "exact_mass":        p.get("ExactMass"),
            "charge":            p.get("Charge"),
        })
    return hits

In [7]:


# ============================================================
# Run similarity search for each seed compound
# ============================================================

print("\nRunning similarity searches (this may take a minute)...")
for record in list(records):   # list() makes a copy so extending is safe
    analogs = find_similar_compounds(record["cid"], record["drug_class"])
    records.extend(analogs)



Running similarity searches (this may take a minute)...
  Similarity search for CID 91801202...
    Found 20 similar CIDs. Fetching properties...
  Similarity search for CID 56944144...
    Found 20 similar CIDs. Fetching properties...
  Similarity search for CID 24965990...
    Found 20 similar CIDs. Fetching properties...
  Similarity search for CID 154617563...
    Found 20 similar CIDs. Fetching properties...


In [8]:

# ============================================================
# 1e. Build DataFrame and save
# ============================================================

df = pd.DataFrame(records)
df = df.drop_duplicates(subset="cid")   # remove any duplicates

print(f"\nLibrary built: {len(df)} compounds total.")
print(df[["name", "drug_class", "molecular_weight", "xlogp", "tpsa"]].head(10))

df.to_csv("orexin_ligand_library_raw.csv", index=False)
print("\nSaved: orexin_ligand_library_raw.csv")




Library built: 80 compounds total.
                             name   drug_class molecular_weight  xlogp  tpsa
0                    Daridorexant         DORA            450.9    4.1  88.9
1                     Lemborexant         DORA            410.4    3.2  77.0
2                      Suvorexant         DORA            450.9    4.9  80.3
3                     Oveporexton      agonist            520.5    3.6  95.1
5    similar_to_91801202_91809208  DORA_analog            487.4    NaN  88.9
6    similar_to_91801202_72694582  DORA_analog            466.9    3.9  88.9
7    similar_to_91801202_72694992  DORA_analog            522.9    4.8  88.9
8   similar_to_91801202_118126995  DORA_analog            432.4    3.2  88.9
9   similar_to_91801202_118127015  DORA_analog            508.0    5.9  67.5
10  similar_to_91801202_118127019  DORA_analog            484.9    3.6  98.2

Saved: orexin_ligand_library_raw.csv


In [9]:
print(f"Number of records: {len(records)}")
print(f"First record: {records[0] if records else 'EMPTY'}")

Number of records: 84
First record: {'name': 'Daridorexant', 'drug_class': 'DORA', 'cid': 91801202, 'molecular_formula': 'C23H23ClN6O2', 'molecular_weight': '450.9', 'iupac_name': '[(2S)-2-(5-chloro-4-methyl-1H-benzimidazol-2-yl)-2-methylpyrrolidin-1-yl]-[5-methoxy-2-(triazol-2-yl)phenyl]methanone', 'canonical_smiles': None, 'isomeric_smiles': None, 'xlogp': 4.1, 'h_bond_donors': 1, 'h_bond_acceptors': 5, 'rotatable_bonds': 4, 'tpsa': 88.9, 'exact_mass': '450.1571017', 'charge': 0}


# Test fetch for just one compound and print everything  (this is a test)
import pubchempy as pcp

name = "Daridorexant"
print(f"Searching PubChem for: {name}")

results = pcp.get_compounds(name, 'name')
print(f"Raw results: {results}")
print(f"Number of results: {len(results)}")

if results:
    c = results[0]
    print(f"CID: {c.cid}")
    print(f"Molecular weight: {c.molecular_weight}")
    print(f"canonical_smiles: {c.canonical_smiles}")
else:
    print("No results returned — PubChem returned empty list")

# Quick confirmation test
results = pcp.get_compounds(
    "Daridorexant", 
    'name',
    properties=['MolecularFormula', 'MolecularWeight', 'IsomericSMILES']
)
c = results[0]
print(c.to_dict())

In [10]:
# Diagnostic — run this as a standalone cell
print("Checking what Python currently knows about...")

try:
    print(f"dora_names: {dora_names}")
except NameError:
    print("dora_names: NOT DEFINED")

try:
    print(f"agonist_names: {agonist_names}")
except NameError:
    print("agonist_names: NOT DEFINED")

try:
    print(f"records: {records}")
except NameError:
    print("records: NOT DEFINED")

try:
    fetch_compound_properties("test", "test")
    print("fetch_compound_properties: DEFINED")
except NameError:
    print("fetch_compound_properties: NOT DEFINED")
except:
    print("fetch_compound_properties: DEFINED")

try:
    find_similar_compounds(0, "test")
    print("find_similar_compounds: DEFINED")
except NameError:
    print("find_similar_compounds: NOT DEFINED")
except:
    print("find_similar_compounds: DEFINED")

Checking what Python currently knows about...
dora_names: ['Daridorexant', 'Lemborexant', 'Suvorexant']
agonist_names: ['Oveporexton']
records: [{'name': 'Daridorexant', 'drug_class': 'DORA', 'cid': 91801202, 'molecular_formula': 'C23H23ClN6O2', 'molecular_weight': '450.9', 'iupac_name': '[(2S)-2-(5-chloro-4-methyl-1H-benzimidazol-2-yl)-2-methylpyrrolidin-1-yl]-[5-methoxy-2-(triazol-2-yl)phenyl]methanone', 'canonical_smiles': None, 'isomeric_smiles': None, 'xlogp': 4.1, 'h_bond_donors': 1, 'h_bond_acceptors': 5, 'rotatable_bonds': 4, 'tpsa': 88.9, 'exact_mass': '450.1571017', 'charge': 0}, {'name': 'Lemborexant', 'drug_class': 'DORA', 'cid': 56944144, 'molecular_formula': 'C22H20F2N4O2', 'molecular_weight': '410.4', 'iupac_name': 'cis-(1R,2S)-2-[(2,4-dimethylpyrimidin-5-yl)oxymethyl]-2-(3-fluorophenyl)-N-(5-fluoro-2-pyridinyl)cyclopropane-1-carboxamide', 'canonical_smiles': None, 'isomeric_smiles': None, 'xlogp': 3.2, 'h_bond_donors': 1, 'h_bond_acceptors': 7, 'rotatable_bonds': 6, 'tp

In [11]:
# Run this as a standalone diagnostic cell
records = []

print("Testing fetch for each seed compound individually...")

for name in ["Daridorexant", "Lemborexant", "Suvorexant", "Oveporexton"]:
    print(f"\n--- Trying: {name} ---")
    try:
        data = fetch_compound_properties(name, drug_class="DORA")
        print(f"  Result: {data}")
        if data:
            records.append(data)
            print(f"  Successfully added. records now has {len(records)} entries.")
        else:
            print(f"  Function returned None.")
    except Exception as e:
        print(f"  CRASHED with error: {e}")

print(f"\nFinal records count: {len(records)}")


Testing fetch for each seed compound individually...

--- Trying: Daridorexant ---
  Result: {'name': 'Daridorexant', 'drug_class': 'DORA', 'cid': 91801202, 'molecular_formula': 'C23H23ClN6O2', 'molecular_weight': '450.9', 'iupac_name': '[(2S)-2-(5-chloro-4-methyl-1H-benzimidazol-2-yl)-2-methylpyrrolidin-1-yl]-[5-methoxy-2-(triazol-2-yl)phenyl]methanone', 'canonical_smiles': None, 'isomeric_smiles': None, 'xlogp': 4.1, 'h_bond_donors': 1, 'h_bond_acceptors': 5, 'rotatable_bonds': 4, 'tpsa': 88.9, 'exact_mass': '450.1571017', 'charge': 0}
  Successfully added. records now has 1 entries.

--- Trying: Lemborexant ---
  Result: {'name': 'Lemborexant', 'drug_class': 'DORA', 'cid': 56944144, 'molecular_formula': 'C22H20F2N4O2', 'molecular_weight': '410.4', 'iupac_name': 'cis-(1R,2S)-2-[(2,4-dimethylpyrimidin-5-yl)oxymethyl]-2-(3-fluorophenyl)-N-(5-fluoro-2-pyridinyl)cyclopropane-1-carboxamide', 'canonical_smiles': None, 'isomeric_smiles': None, 'xlogp': 3.2, 'h_bond_donors': 1, 'h_bond_accep